In [16]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import pytidycensus as tc
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Literal
from qbstyles import mpl_style
import glob

%config InlineBackend.figure_format = 'retina'

mpl_style(dark=True)


from pyresearch import collect_acs_data


In [3]:
load_dotenv()

API_KEY: str = os.getenv("CENSUS_KEY")
DATA: str = os.getenv("DATA")


## RUCC Codes

|Code|Description|Category|
|---|---|---|
|1|Counties in metro areas of 1 million population or more|Metropolitan|
|2|Counties in metro areas of 250,000 to 1 million population|Metropolitan|
|3|Counties in metro areas of fewer than 250,000 population|Metropolitan|
|4|Urban population of 20,000 or more, adjacent to a metro area|Nonmetropolitan|
|5|Urban population of 20,000 or more, not adjacent to a metro area|Nonmetropolitan|
|6|Urban population of 5,000 to 20,000, adjacent to a metro area|Nonmetropolitan|
|7|Urban population of 5,000 to 20,000, not adjacent to a metro area|Nonmetropolitan|
|8|Urban population of fewer than 5,000, adjacent to a metro area|Nonmetropolitan|
|9|Urban population of fewer than 5,000, not adjacent to a metro area|Nonmetropolitan|


In [13]:
def collect_rucc_data(data_path: str) -> pd.DataFrame:
    data = pd.read_csv(f"{data_path}/RUCC.csv")
    data.drop(columns=["Population_2020", "Description"], inplace=True)
    data.rename(columns={"RUCC_2023": "RUCC"}, inplace=True)
    data["County_Name"] = data["County_Name"].str.replace("County", "").str.strip()
    data["County_Name"] = data["County_Name"].str.replace("Parish", "").str.strip()

    data.rename(columns={"State": "state", "County_Name": "county"}, inplace=True)

    return data


rucc = collect_rucc_data(data_path=DATA)
rucc.head()


,FIPS,state,county,RUCC
0,1001,AL,Autauga,2.0
1,1003,AL,Baldwin,3.0
2,1005,AL,Barbour,6.0
3,1007,AL,Bibb,1.0
4,1009,AL,Blount,1.0


In [5]:
vars = {
    "pov_under_50": "C17002_002E",
    "pov_under_100": "C17002_003E",
    "white": "B01001A_001E",
    "black": "B01001B_001E",
    "total_pop": "B01003_001E",
    "am_in_ala_nat": "B01001C_001E",
    "asian": "B01001D_001E",
    "haw_pac": "B01001E_001E",
    "other": "B01001F_001E",
    "hisp_lat": "B01001I_001E",
    "male_no_hs": "C15002A_003E",
    "fem_no_hs": "C15002A_008E",
    "not_citizen": "B05001_006E",
    "median_earnings": "B08521_001E",
    "gini_index": "B19083_001E",
}


In [6]:
# Get census data for each state

states_to_remove = [
    "PR",
    "HI",
    "GU",
    "MP",
    "AS",
    "AK",
    "VI",
]  # Filter states to continental US + D.C.

census = collect_acs_data(
    API_KEY=API_KEY,
    geography="county",
    year=2019,
    states=rucc["state"], 
    states_to_remove=states_to_remove,
    vars=vars,
    max_workers=10
    )

census["poverty"] = census["pov_under_50"] + census["pov_under_100"]
census["no_hs"] = census["male_no_hs"] + census["fem_no_hs"]

scale_to_pop = [
    "poverty",
    "no_hs",
    "white",
    "black",
    "am_in_ala_nat",
    "asian",
    "haw_pac",
    "other",
    "hisp_lat",
    "not_citizen",
]

for var in scale_to_pop:
    census[var] = round(census[var] / census["total_pop"], 2)

census.drop(
    columns=[
        "pov_under_50",
        "pov_under_100",
        "male_no_hs",
        "fem_no_hs",
        "county",
        "state",
    ],
    inplace=True,
)
census.head()


Census API key has been set for this session.
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting data from the 2019-2023 5-year ACS
Getting 

,GEOID,white,black,total_pop,am_in_ala_nat,asian,haw_pac,other,hisp_lat,not_citizen,median_earnings,gini_index,NAME,poverty,no_hs
0,20001,0.91,0.02,12491,0.01,0.01,0.01,0.01,0.04,0.02,40856.0,0.4012,"Allen County, Kansas",0.13,0.04
1,20003,0.92,0.01,7819,0.00,0.00,0.00,0.00,0.02,0.00,39668.0,0.3977,"Anderson County, Kansas",0.11,0.07
2,20005,0.87,0.04,16211,0.00,0.01,0.00,0.01,0.04,0.01,39104.0,0.4313,"Atchison County, Kansas",0.12,0.03
3,20007,0.93,0.00,4153,0.02,0.01,0.00,0.01,0.06,0.01,38717.0,0.4462,"Barber County, Kansas",0.17,0.04
4,20009,0.84,0.01,25275,0.01,0.00,0.00,0.03,0.16,0.02,38957.0,0.4261,"Barton County, Kansas",0.14,0.04


## ICE Detention Data

### Overview

From [dataset Github](https://github.com/vera-institute/ice-detention-trends/tree/main#about-the-data)

> Vera’s ICE Detention Trends dashboard reveals an unprecedented level of detail about ICE detention populations—nationally and across the 1,464 facilities in which ICE detained people—on each day of the 16-year period from fiscal year 2009 through the beginning of fiscal year 2026 (October 1, 2008, through October 15, 2025). This repository includes the aggregated data visualized in the dashboard, including information on:
>
> - Midnight population: the daily number of people detained at midnight (nationally and by facility).
> - 24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight (nationally and by facility). While ICE relies solely on midnight populations in its reporting, Vera includes both types of daily populations—midnight and 24-hour—as the two can differ drastically.
> - Book-ins: the daily number of people ICE booked into custody (nationally).
> - Book-outs: the daily number of people ICE booked out of custody (nationally).
> - Facility names, locations, and types (as coded by ICE in other datasets, where available).
> 
> The original datasets included facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility > locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all 1,464 facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type. To simplify map filtering options, Vera grouped facility types assigned by ICE, as well as ones manually entered by Vera, into the following categories:
>
> - Non-Dedicated: IGSA (Inter-governmental Service Agreement).
> - Dedicated: DIGSA (Dedicated IGSA), SPC (Service Processing Center), CDF (Contract Detention Facility).
> - Federal: USMS IGA (U.S. Marshals Service Inter-governmental Agreement), BOP (Bureau of Prisons), USMS CDF (U.S. Marshals Service Contract Detention Facility), DOD (Department of Defense), MOC (Migrant Operations Center). Because ICE can be added to other federal agencies’ facility contracts or agreements through a “rider,” Vera reports federal facilities as a separate category, rather than grouped with other categories such as non-dedicated facilities.
> - Hold/Staging.
> - Family/Youth: Family, Family Staging, Juvenile. ICE’s use of the “Juvenile” facility type reflects ICE detention and does not refer to facilities used to detain unaccompanied children in the custody of the Office of Refugee Resettlement (ORR).
> - Medical: Facilities coded by ICE as “Hospital” and medical or mental health facilities manually coded by Vera.
> - Hotel: Facilities coded by ICE as “Hotel” and facilities manually coded by Vera.
> - Other/Unknown: Facilities coded by ICE as “Other” or ones for which Vera was unable to assign facility type.


### Facilities

#### Variables

> The ICE detention datasets include facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type.


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|latitude|`numeric`|The latitude coordinate of the facility location|
|longitude|`numeric`|The longitude coordinate of the facility location|
|address|`string`|The best known facility address|
|city|`string`|The city in which the facility is located|
|county|`string`|The county in which the facility is located|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|aor|`aor`|The ICE Area of Responsibility (AOR), originally mapped by Will Craft of the Guardian US. This reflects county boundaries extracted from ICE's [field office map](https://www.ice.gov/doclib/about/offices/ero/pdf/eroFieldOffices.pdf), last updated by ICE in February 2024.|


In [18]:
if "facilities" not in locals():
    print("Processing facilities data")
    facilities = pd.read_csv(f"{DATA}/ice-detention-trends/metadata/facilities.csv")
    facilities.drop(columns=["address", "city", "zip"], inplace=True)

    facilities.rename(columns={"FIPS": "GEOID"}, inplace=True)
    facilities.drop(columns=["detention_facility_name", "state"], inplace=True)
    facilities.sort_values("county", inplace=True)
else:
    print("facilities data already processed")
facilities.head()


Processing facilities data


,detention_facility_code,county,aor,latitude,longitude,type_detailed,type_grouped
126,BOIHOLD,Ada,Salt Lake City,43.592766,-116.286069,Hold,Hold/Staging
6,ADACOID,Ada,Salt Lake City,43.606755,-116.269811,IGSA,Non-Dedicated
7,ADAIRKY,Adair,Chicago,37.103848,-85.306634,Other,Other/Unknown
114,BIINCCO,Adams,Denver,39.760890,-104.849106,Unknown,Other/Unknown
395,DENICDF,Adams,Denver,39.766325,-104.864502,CDF,Dedicated


### State Data

Facility-level population statistics for each day between October 1, 2008, and October 15, 2025, including midnight population and 24-hour population. 


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|date|`date`|The day each count is reported for (`yyyy-mm-dd` format)|
|daily_pop|`numeric`|24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight|
|midnight_pop|`numeric`|Midnight population: the daily number of people detained at midnight|


In [19]:
if "state_data" not in locals():
    print("Processing state detention data")
    state_dir = f"{DATA}/ice-detention-trends/facilities/by_state"
    file_pattern = os.path.join(state_dir, "*.csv")
    csvs = glob.glob(file_pattern)
    state_data = pd.concat(
        [pd.read_csv(csv) for csv in csvs], ignore_index=True, sort=False
    )
    state_data["date"] = pd.to_datetime(state_data["date"])

else:
    print("state detention data already processed")


Processing state detention data


In [22]:

# missing_counties = state_data["county"].isna().sum()
# missing_states = state_data["state"].isna().sum()

# print(
#     f"Total Observations: {len(state_data)},\n Missing states: {missing_states},\n Missing counties: {missing_counties}"
# )
cols_to_fix = [
    "detention_facility_code",
    "detention_facility_name",
    # "state",
    "type_detailed",
    "type_grouped",
]

# This converts them to the actual 'string' Dtype
state_data[cols_to_fix] = state_data[cols_to_fix].astype("string")

cols = ["daily_pop", "midnight_pop"]

# apply pd.to_numeric to each column individually
state_data[cols] = state_data[cols].apply(
    pd.to_numeric, errors="coerce", downcast="integer"
)

print(state_data.info())
state_data.head()


KeyError: "['type_detailed', 'type_grouped'] not in index"

## Incarceration Trends Data

See [Incarceration Trends Dataset](https://github.com/vera-institute/incarceration-trends)


In [12]:
jail_construction = pd.read_csv(f'{DATA}/incarceration_trends_jail_construction.csv')
jail_construction.head()


,jid,fips,county_name,state_abbr,project_year,project_type,project_status,source_1,source_2,source_3,source_4,source_5,source_6,source_7,source_8,project_amount,jail_capacity_before,jail_capacity_after
0,1003001,1003,Baldwin County,AL,2020,New Jail,Passed,https://perma.cc/JQR3-5PTC,0,0,0,0,0,0,0,65000000.0,650.0,900.0
1,1009001,1009,Blount County,AL,2020,Jail Expansion,Passed,https://perma.cc/W8CS-GN5P,https://perma.cc/R4J4-45KH,0,0,0,0,0,0,2400000.0,101.0,172.0
2,1013001,1013,Butler,AL,2008,New Jail,Passed,https://perma.cc/7EE3-AE3B,https://perma.cc/GL7E-A7AG,0,0,0,0,0,0,6000000.0,53.0,94.0
3,1015002,1015,Calhoun County,AL,2020,Jail Expansion,Passed,https://perma.cc/5SZ3-2268,0,0,0,0,0,0,0,6000000.0,NaN,NaN
4,1017002,1017,Chambers County,AL,2017,Jail Expansion,Passed,https://perma.cc/RW65-KSRY,0,0,0,0,0,0,0,4700000.0,135.0,256.0


(245840, 0)